# Module 11: Model Lifecycle Management

**Snowpark ML Fundamentals — Week 3 | Lunch & Learn**

---

## Learning Objectives
- Use aliases for zero-downtime model promotion
- Run inference by alias instead of version name
- Explore `predict_proba` and other model functions
- Attach tags and comments for governance

## Key Concept
> **Aliases** decouple deployment from versioning. Your inference code always
> references `'production'` — when you promote a new version, the alias moves
> and all consumers automatically get the new model.

## Prerequisites
This notebook assumes `CHURN_PREDICTOR_W3` with V1 and V2 exists from notebook 10.

---

In [1]:
%load_ext autoreload
%autoreload 2

## 11.1 Setup

In [2]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.preprocessing.transformers import build_preprocessing_pipeline
from snowpark_fundamentals.registry.model_registry import (
    get_registry,
    list_versions,
    set_model_alias,
    get_model_by_alias,
    show_model_functions,
    predict_proba,
    set_model_tags,
    set_model_comment,
)

session = create_session(env_path='../.env')

current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

# Prepare test data for inference demos
NUMERIC_COLS = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
FEATURE_COLS = [f"{c}_SCALED" for c in NUMERIC_COLS] + [f"{c}_ENCODED" for c in CATEGORICAL_COLS]

df = create_customer_churn_dataset(session, n_rows=5000)
df_processed, _ = build_preprocessing_pipeline(df, NUMERIC_COLS, CATEGORICAL_COLS)
_, test_df = split_data(df_processed)

print("Setup complete. Registry connected.")
display(versions := list_versions(registry, 'CHURN_PREDICTOR_W3'))

Setup complete. Registry connected.


,created_on,name,aliases,comment,database_name,schema_name,model_name,is_default_version,functions,metadata,user_data,model_attributes,size,environment,runnable_in,inference_services
0,2026-03-10 20:52:31.579000-04:00,V1,"[""FIRST""]",None,MLDS_Q,PREDICTIONS,CHURN_PREDICTOR_W3,false,"[""PREDICT_PROBA"",""EXPLAIN"",""PREDICT""]","{""metrics"": {""accuracy"": 0.819846, ""precision""...",{},"{""framework"":""xgboost"",""task"":""TABULAR_BINARY_...",325380,"{""default"":{""python_version"":""3.10"",""cuda_vers...","[""WAREHOUSE"",""SNOWPARK_CONTAINER_SERVICES""]",[]
1,2026-03-10 20:52:49.736000-04:00,V2,"[""DEFAULT"",""LAST""]",None,MLDS_Q,PREDICTIONS,CHURN_PREDICTOR_W3,true,"[""PREDICT_PROBA"",""EXPLAIN"",""PREDICT""]","{""metrics"": {""accuracy"": 0.842004, ""precision""...",{},"{""framework"":""xgboost"",""task"":""TABULAR_BINARY_...",989520,"{""default"":{""python_version"":""3.10"",""cuda_vers...","[""WAREHOUSE"",""SNOWPARK_CONTAINER_SERVICES""]",[]


## 11.2 Alias Workflow

Aliases are **movable labels** that point to exactly one version at a time.
When you assign an alias to a new version, it's automatically removed from the old one.

Common aliases:
- `production` — the version serving live traffic
- `staging` — candidate for next promotion
- `archived` — retired versions kept for audit

In [3]:
# Assign V1 as the production model
set_model_alias(registry, 'CHURN_PREDICTOR_W3', 'V1', 'production')
print("V1 aliased as 'production'")

# Check aliases
versions_df = list_versions(registry, 'CHURN_PREDICTOR_W3')
print(versions_df[['name', 'aliases']].to_string())

V1 aliased as 'production'
  name                 aliases
0   V1  ["PRODUCTION","FIRST"]
1   V2      ["DEFAULT","LAST"]


## 11.3 Inference by Alias

This is the **key production pattern**: always reference models by alias.
When you promote a new version, this code doesn't change at all.

In [4]:
# Load the 'production' model — no version number in this code!
production_model = get_model_by_alias(registry, 'CHURN_PREDICTOR_W3', 'production')

# Run inference
predictions = production_model.run(test_df.select(FEATURE_COLS), function_name="predict")
print("Predictions from 'production' alias (currently V1):")
predictions.show(5)

Predictions from 'production' alias (currently V1):
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"output_feature_0"  |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|-1.4536816815  |0.523941485064372       |0.5577318136234114        |0.8146184680621967      |-0.3151768044             

## 11.4 Promote V2 to Production

Moving the `production` alias to V2 is a **zero-downtime promotion**.
The alias automatically detaches from V1. We can also explicitly tag V1 as `archived`.

In [5]:
# Promote V2 — the production alias moves automatically
set_model_alias(registry, 'CHURN_PREDICTOR_W3', 'V2', 'production')
print("V2 is now 'production'")

# Archive V1
set_model_alias(registry, 'CHURN_PREDICTOR_W3', 'V1', 'archived')
print("V1 is now 'archived'")

# Verify — same inference code, now uses V2
versions_df = list_versions(registry, 'CHURN_PREDICTOR_W3')
print(versions_df[['name', 'aliases']].to_string())

V2 is now 'production'
V1 is now 'archived'
  name                          aliases
0   V1             ["ARCHIVED","FIRST"]
1   V2  ["PRODUCTION","DEFAULT","LAST"]


## 11.5 predict vs predict_proba

Registered XGBoost models expose multiple callable functions.
`show_model_functions()` lists them. `predict_proba` returns class
probabilities — essential for threshold tuning and calibration.

In [6]:
# List available functions
functions = show_model_functions(registry, 'CHURN_PREDICTOR_W3', 'V2')
print("Available functions:", functions)

Available functions: [{'name': 'EXPLAIN', 'target_method': 'explain', 'target_method_function_type': 'TABLE_FUNCTION', 'signature': ModelSignature(
    inputs=[
        FeatureSpec(dtype=DataType.DOUBLE, name='AGE_SCALED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='TENURE_MONTHS_SCALED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='MONTHLY_CHARGES_SCALED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='TOTAL_CHARGES_SCALED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='SUPPORT_TICKETS_SCALED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='USAGE_HOURS_PER_WEEK_SCALED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='NUM_DEPENDENTS_SCALED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='PLAN_TYPE_ENCODED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='CONTRACT_TYPE_ENCODED', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name

In [7]:
# Reload the production model (now V2 after promotion in 11.4)
production_model = get_model_by_alias(registry, 'CHURN_PREDICTOR_W3', 'production')

# predict: returns class labels (0 or 1)
labels = production_model.run(test_df.select(FEATURE_COLS), function_name="predict")
print("predict() — class labels:")
labels.show(5)

# predict_proba: returns probability scores per class
probas = predict_proba(registry, 'CHURN_PREDICTOR_W3', 'V2', test_df.select(FEATURE_COLS))
print("predict_proba() — probability scores:")
probas.show(5)

predict() — class labels:
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"output_feature_0"  |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|-1.4536816815  |0.523941485064372       |0.5577318136234114        |0.8146184680621967      |-0.3151768044             |1.4348930167             

## 11.6 Tags & Metadata

Tags and comments are governance primitives:
- **Tags**: key-value pairs for filtering and discovery (team, project, environment)
- **Comments**: free-text descriptions on models or versions

> **Note**: Snowflake tags are **schema-level objects** — they must be created
> with `CREATE TAG` before they can be assigned to a model.

In [8]:
# Create tags in the schema (tags are schema-level objects in Snowflake)
for tag_name in ['team', 'project', 'environment']:
    session.sql(f"CREATE TAG IF NOT EXISTS {current_db}.{current_schema}.{tag_name}").collect()
print("Tags created.")

# Set tags for governance and discovery
set_model_tags(registry, 'CHURN_PREDICTOR_W3', {
    'team': 'ml-platform',
    'project': 'churn',
    'environment': 'development',
})
print("Tags set on CHURN_PREDICTOR_W3")

# Set model-level comment
set_model_comment(
    registry, 'CHURN_PREDICTOR_W3',
    'XGBoost binary classifier for customer churn prediction. '
    'Trained on synthetic data for Week 3 tutorial.',
)

# Set version-level comment
set_model_comment(
    registry, 'CHURN_PREDICTOR_W3',
    'Tuned: n_estimators=200, max_depth=8, lr=0.05. Promoted to production.',
    version_name='V2',
)
print("Comments set on model and V2")

Tags created.
Tags set on CHURN_PREDICTOR_W3
Comments set on model and V2


## Key Takeaways

1. **Aliases** decouple inference code from version numbers
2. Moving an alias is a **zero-downtime promotion** — no redeployment needed
3. `predict_proba` returns probabilities for **threshold tuning**
4. **Tags** enable governance: filter by team, project, environment
5. **Comments** provide human-readable context for each model and version

---

**Next: [12 — Feature Store to Registry](12_feature_store_to_registry.ipynb)** — Train on Feature Store data, register, and serve with feature retrieval

In [9]:
session.close()